# Hedonic Pricing Analysis (Improved)

This notebook rebuilds the hedonic analysis with a cleaner workflow and fixes several issues in the original version:

- removes a perfectly collinear category overlap between `flat_type == 7` and `flat_model == "Multi Generation"`
- treats the model with accessibility and general school-density controls as the preferred specification
- fixes the restricted-sample robustness check by using numeric `flat_type` codes (`4` and `5`)
- reports effect sizes in percent terms for easier interpretation
- keeps clustered-by-town standard errors as a sensitivity check rather than the headline inference

The design remains observational. Coefficients should be interpreted as conditional associations, not causal effects.

In [62]:
import warnings

import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
from statsmodels.tools.sm_exceptions import ValueWarning

warnings.filterwarnings("ignore", category=ValueWarning)
pd.set_option("display.max_columns", 200)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

In [63]:
df = pd.read_csv('../data/final_df.csv')
df['year_quarter'] = df['year'].astype(int).astype(str) + 'Q' + df['quarter'].astype(int).astype(str)

print('Rows, columns:', df.shape)
print('Flat type codes:', sorted(df['flat_type'].dropna().unique().tolist()))
print('Towns:', df['town'].nunique())
print('Year-quarter periods:', df['year_quarter'].nunique())

df.head()

Rows, columns: (291419, 51)
Flat type codes: [1, 2, 3, 4, 5, 6, 7]
Towns: 26
Year-quarter periods: 52


,month,town,flat_type,block,street_name,floor_area_sqm,flat_model,lease_commence_date,resale_price,year,quarter,remaining_lease_years,full_address,log_resale_price,lat,lon,floor_area_sqft,price_per_sqft,log_price,log_price_per_sqft,mature_estate,storey_relative_category,Index,real_price,real_price_psf,log_real_price,log_real_price_psf,dist_to_nearest_hawker_km,dist_to_nearest_busstop_km,dist_to_nearest_mall_km,dist_to_nearest_mrt_km,dist_to_cbd_km,countall_0_1km,countall_1_2km,count_0_1km_good75,count_1_2km_good75,d_0_1km_good75,d_1_2km_good75,count_0_1km_good80,count_1_2km_good80,d_0_1km_good80,d_1_2km_good80,count_0_1km_good85,count_1_2km_good85,d_0_1km_good85,d_1_2km_good85,count_0_1km_good90,count_1_2km_good90,d_0_1km_good90,d_1_2km_good90,year_quarter
0,2013-01-01,ANG MO KIO,2,510,ANG MO KIO AVE 8,44.0000,Improved,1980,"253,000.0000",2013,1,65.9979,510 ANG MO KIO AVE 8,12.4411,1.3734,103.8491,473.6116,534.1930,12.4411,6.2808,1,LOW_IN_ESTATE,148.6000,"170,255.7201",359.4838,12.0451,5.8847,0.3160,0.1867,0.2303,0.3894,9.9305,3,4,0,1,0,1,0,1,0,1,0,1,0,1,0,1,0,1,2013Q1
1,2013-01-01,ANG MO KIO,2,314,ANG MO KIO AVE 3,44.0000,Improved,1978,"270,000.0000",2013,1,63.9993,314 ANG MO KIO AVE 3,12.5062,1.3662,103.8501,473.6116,570.0874,12.5062,6.3458,1,LOW_IN_ESTATE,148.6000,"181,695.8277",383.6389,12.1101,5.9497,0.3124,0.1202,0.4069,0.4162,9.1305,3,7,0,3,0,1,0,3,0,1,0,3,0,1,0,3,0,1,2013Q1
2,2013-01-01,ANG MO KIO,2,323,ANG MO KIO AVE 3,44.0000,Improved,1977,"283,000.0000",2013,1,63.0000,323 ANG MO KIO AVE 3,12.5532,1.3679,103.8477,473.6116,597.5360,12.5532,6.3928,1,LOW_IN_ESTATE,148.6000,"190,444.1454",402.1104,12.1571,5.9967,0.4267,0.1089,0.1845,0.3044,9.3255,3,6,0,3,0,1,0,3,0,1,0,3,0,1,0,3,0,1,2013Q1
3,2013-01-01,ANG MO KIO,3,170,ANG MO KIO AVE 4,61.0000,Improved,1986,"305,000.0000",2013,1,71.9993,170 ANG MO KIO AVE 4,12.6281,1.3740,103.8364,656.5979,464.5156,12.6281,6.1410,1,MID_IN_ESTATE,148.6000,"205,248.9906",312.5946,12.2320,5.7449,0.3107,0.1364,1.0838,0.2826,10.1324,3,4,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,2013Q1
4,2013-01-01,ANG MO KIO,3,174,ANG MO KIO AVE 4,60.0000,Improved,1986,"320,000.0000",2013,1,71.9993,174 ANG MO KIO AVE 4,12.6761,1.3751,103.8376,645.8340,495.4834,12.6761,6.2055,1,LOW_IN_ESTATE,148.6000,"215,343.2032",333.4343,12.2800,5.8094,0.1843,0.0698,0.9916,0.4207,10.2320,3,4,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,2013Q1


In [64]:
# check unique flat_type for flat_model = Multi Generation
multi_gen_flat_types = df[df['flat_model'] == 'Multi Generation']['flat_type'].unique()
print('Unique flat types for Multi Generation flat model:', multi_gen_flat_types)

Unique flat types for Multi Generation flat model: [7]


In [65]:
pd.crosstab(df["flat_type"], df["flat_model"])

flat_model,2-room,3Gen,Adjoined flat,Apartment,DBSS,Improved,Improved-Maisonette,Maisonette,Model A,Model A-Maisonette,Model A2,Multi Generation,New Generation,Premium Apartment,Premium Apartment Loft,Premium Maisonette,Simplified,Standard,Terrace,Type S1,Type S2
flat_type,,,,,,,,,,,,,,,,,,,,,
1,0,0,0,0,0,120,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
2,365,0,0,0,2,877,0,0,2906,0,0,0,0,57,0,0,0,1016,0,0,0
3,0,0,0,0,503,18168,0,0,19336,0,0,0,25569,1399,0,0,3314,4313,146,0,0
4,0,0,50,0,1525,3313,0,0,73550,0,3588,0,14013,16047,95,0,8974,113,12,498,0
5,0,66,260,0,1806,49690,38,0,3681,503,0,0,0,10918,31,0,0,3055,0,0,241
6,0,0,166,10623,0,0,0,7996,0,0,0,0,0,2355,0,22,0,0,0,0,0
7,0,0,0,0,0,0,0,0,0,0,0,99,0,0,0,0,0,0,0,0,0


In [66]:
pd.crosstab(df["flat_type"], df["town"])
pd.crosstab(df["flat_model"], df["town"])

town,ANG MO KIO,BEDOK,BISHAN,BUKIT BATOK,BUKIT MERAH,BUKIT PANJANG,BUKIT TIMAH,CENTRAL AREA,CHOA CHU KANG,CLEMENTI,GEYLANG,HOUGANG,JURONG EAST,JURONG WEST,KALLANG/WHAMPOA,MARINE PARADE,PASIR RIS,PUNGGOL,QUEENSTOWN,SEMBAWANG,SENGKANG,SERANGOON,TAMPINES,TOA PAYOH,WOODLANDS,YISHUN
flat_model,,,,,,,,,,,,,,,,,,,,,,,,,,
2-room,0,0,0,58,4,5,0,0,13,0,1,35,4,7,0,0,9,31,1,63,81,0,2,1,11,39
3Gen,0,0,0,6,0,0,0,0,0,0,0,11,0,13,0,0,0,3,0,0,11,0,5,0,3,14
Adjoined flat,109,49,2,11,39,1,2,13,4,28,30,13,8,14,29,8,1,0,47,0,0,9,12,19,10,18
Apartment,73,376,57,416,0,405,8,0,687,18,49,495,247,1186,75,0,1735,0,39,574,1035,280,837,55,1543,433
DBSS,313,224,241,0,0,0,0,0,0,344,0,250,0,214,360,0,151,0,0,0,0,0,857,555,0,327
Improved,1632,4192,1241,1788,3734,2266,339,1098,3374,650,1707,3133,1366,5840,3370,1374,2501,3090,3064,2199,5371,1471,4355,4632,5524,2857
Improved-Maisonette,0,0,0,0,0,0,0,0,0,0,0,38,0,0,0,0,0,0,0,0,0,0,0,0,0,0
Maisonette,18,357,476,507,0,363,129,0,385,169,202,1015,321,646,94,0,459,0,27,23,69,508,1031,165,545,487
Model A,1251,2089,1528,5120,4326,4565,156,322,6214,1300,1866,5422,1620,7752,2473,0,3707,5987,2704,3822,7917,1022,10551,2732,6994,8033


In [67]:
pd.crosstab(df["town"], df["d_0_1km_good80"])

d_0_1km_good80,0,1
town,,
ANG MO KIO,9265,3234
BEDOK,7703,8539
BISHAN,562,4587
BUKIT BATOK,7792,4177
BUKIT MERAH,10582,612
BUKIT PANJANG,10097,297
BUKIT TIMAH,0,690
CENTRAL AREA,2146,210
CHOA CHU KANG,6747,6146


## Data checks

The original notebook produced unstable estimates because the rare `flat_type == 7` category is perfectly aligned with `flat_model == "Multi Generation"`. Keeping both sets of dummies creates a singular design matrix and leads to nonsensical coefficients and standard errors for some models.

We drop those rows before estimation. This affects only a tiny fraction of the sample and removes the exact overlap.

In [68]:
#overlap_mask = (df['flat_type'] == 7) & (df['flat_model'] == 'Multi Generation')
#print('Rows with exact overlap:', int(overlap_mask.sum()))

#df_clean = df.loc[~overlap_mask].copy()
#print('Cleaned shape:', df_clean.shape)

In [69]:
df.columns

Index(['month', 'town', 'flat_type', 'block', 'street_name', 'floor_area_sqm',
       'flat_model', 'lease_commence_date', 'resale_price', 'year', 'quarter',
       'remaining_lease_years', 'full_address', 'log_resale_price', 'lat',
       'lon', 'floor_area_sqft', 'price_per_sqft', 'log_price',
       'log_price_per_sqft', 'mature_estate', 'storey_relative_category',
       'Index', 'real_price', 'real_price_psf', 'log_real_price',
       'log_real_price_psf', 'dist_to_nearest_hawker_km',
       'dist_to_nearest_busstop_km', 'dist_to_nearest_mall_km',
       'dist_to_nearest_mrt_km', 'dist_to_cbd_km', 'countall_0_1km',
       'countall_1_2km', 'count_0_1km_good75', 'count_1_2km_good75',
       'd_0_1km_good75', 'd_1_2km_good75', 'count_0_1km_good80',
       'count_1_2km_good80', 'd_0_1km_good80', 'd_1_2km_good80',
       'count_0_1km_good85', 'count_1_2km_good85', 'd_0_1km_good85',
       'd_1_2km_good85', 'count_0_1km_good90', 'count_1_2km_good90',
       'd_0_1km_good90', 'd_1_2km_g

In [70]:
flat_model_dummies = pd.get_dummies(df["flat_model"], prefix="fm", drop_first=True)

# Remove the problematic column
flat_model_dummies = flat_model_dummies.drop(columns=["fm_Multi Generation"])


In [71]:
flat_type_dummies = pd.get_dummies(df["flat_type"], prefix="ft", drop_first=True)

df_clean = pd.concat([
    df,
    flat_type_dummies,
    flat_model_dummies
], axis=1)

In [72]:
flat_model_variables = flat_model_dummies.columns.tolist()
flat_type_variables = flat_type_dummies.columns.tolist()

In [73]:
flat_model_variables

['fm_3Gen',
 'fm_Adjoined flat',
 'fm_Apartment',
 'fm_DBSS',
 'fm_Improved',
 'fm_Improved-Maisonette',
 'fm_Maisonette',
 'fm_Model A',
 'fm_Model A-Maisonette',
 'fm_Model A2',
 'fm_New Generation',
 'fm_Premium Apartment',
 'fm_Premium Apartment Loft',
 'fm_Premium Maisonette',
 'fm_Simplified',
 'fm_Standard',
 'fm_Terrace',
 'fm_Type S1',
 'fm_Type S2']

In [74]:
flat_type_variables

['ft_2', 'ft_3', 'ft_4', 'ft_5', 'ft_6', 'ft_7']

In [75]:
main_school_var = 'd_0_1km_good80'
secondary_school_var = 'd_1_2km_good80'

base_vars = [
    'log_real_price_psf',
    main_school_var,
    secondary_school_var,
    'countall_0_1km',
    'countall_1_2km',
    'floor_area_sqm',
    'remaining_lease_years',
    'mature_estate',
    'dist_to_cbd_km',
    'dist_to_nearest_mrt_km',
    'dist_to_nearest_mall_km',
    'dist_to_nearest_hawker_km',
    'dist_to_nearest_busstop_km',
    'town',
    'storey_relative_category',
    'year_quarter',
    'fm_3Gen',
    'fm_Adjoined flat',
    'fm_Apartment',
    'fm_DBSS',
    'fm_Improved',
    'fm_Improved-Maisonette',
    'fm_Maisonette',
    'fm_Model A',
    'fm_Model A-Maisonette',
    'fm_Model A2',
    'fm_New Generation',
    'fm_Premium Apartment',
    'fm_Premium Apartment Loft',
    'fm_Premium Maisonette',
    'fm_Simplified',
    'fm_Standard',
    'fm_Terrace',
    'fm_Type S1',
    'fm_Type S2',
    'ft_2', 'ft_3', 'ft_4', 'ft_5', 'ft_6', 'ft_7'
]

reg_df = df_clean[base_vars].dropna().copy()
print('Regression sample:', reg_df.shape)
reg_df[[
    'log_real_price_psf',
    main_school_var,
    secondary_school_var,
    'countall_0_1km',
    'countall_1_2km',
    'floor_area_sqm',
    'remaining_lease_years',
]].describe().T

Regression sample: (291419, 41)


,count,mean,std,min,25%,50%,75%,max
log_real_price_psf,"291,419.0000",5.7565,0.2184,4.8181,5.6067,5.7210,5.8659,6.8012
d_0_1km_good80,"291,419.0000",0.4226,0.4940,0.0000,0.0000,0.0000,1.0000,1.0000
d_1_2km_good80,"291,419.0000",0.5794,0.4937,0.0000,0.0000,1.0000,1.0000,1.0000
countall_0_1km,"291,419.0000",3.0681,1.5703,0.0000,2.0000,3.0000,4.0000,10.0000
countall_1_2km,"291,419.0000",5.4999,2.7204,0.0000,4.0000,5.0000,7.0000,15.0000
floor_area_sqm,"291,419.0000",96.6506,24.1441,31.0000,76.0000,93.0000,112.0000,366.7000
remaining_lease_years,"291,419.0000",74.2300,13.4618,40.0833,63.2519,73.5833,85.7500,97.8364


In [76]:
treatment_summary = pd.DataFrame({
    'share': [
        reg_df[main_school_var].mean(),
        reg_df[secondary_school_var].mean(),
    ],
    'mean_log_price_psf': [
        reg_df.loc[reg_df[main_school_var] == 1, 'log_real_price_psf'].mean(),
        reg_df.loc[reg_df[secondary_school_var] == 1, 'log_real_price_psf'].mean(),
    ],
}, index=['good80 within 0-1km', 'good80 within 1-2km'])

treatment_summary

,share,mean_log_price_psf
good80 within 0-1km,0.4226,5.7644
good80 within 1-2km,0.5794,5.7763


## Model setup

Model 1 uses structural controls and fixed effects.

Model 2 adds linear accessibility controls.

Model 3 adds general school-density controls and is the preferred specification because it compares homes near a good school to homes with similar overall school density.

In [77]:
formula_m1 = f'''
log_real_price_psf ~ {main_school_var} + {secondary_school_var}
+ floor_area_sqm
+ remaining_lease_years
+ mature_estate

+ fm_3Gen
+ Q("fm_Adjoined flat")
+ fm_Apartment
+ fm_DBSS
+ fm_Improved
+ Q("fm_Improved-Maisonette")
+ fm_Maisonette
+ Q("fm_Model A")
+ Q("fm_Model A-Maisonette")
+ Q("fm_Model A2")
+ Q("fm_New Generation")
+ Q("fm_Premium Apartment")
+ Q("fm_Premium Apartment Loft")
+ Q("fm_Premium Maisonette")
+ fm_Simplified
+ fm_Standard
+ fm_Terrace
+ Q("fm_Type S1")
+ Q("fm_Type S2")

+ ft_2 + ft_3 + ft_4 + ft_5 + ft_6 + ft_7

+ C(storey_relative_category)
+ C(town)
+ C(year_quarter)
'''

formula_m2 = f'''
log_real_price_psf ~ {main_school_var} + {secondary_school_var}
+ floor_area_sqm
+ remaining_lease_years
+ mature_estate
+ dist_to_cbd_km
+ dist_to_nearest_mrt_km
+ dist_to_nearest_mall_km
+ dist_to_nearest_hawker_km
+ dist_to_nearest_busstop_km

+ fm_3Gen
+ Q("fm_Adjoined flat")
+ fm_Apartment
+ fm_DBSS
+ fm_Improved
+ Q("fm_Improved-Maisonette")
+ fm_Maisonette
+ Q("fm_Model A")
+ Q("fm_Model A-Maisonette")
+ Q("fm_Model A2")
+ Q("fm_New Generation")
+ Q("fm_Premium Apartment")
+ Q("fm_Premium Apartment Loft")
+ Q("fm_Premium Maisonette")
+ fm_Simplified
+ fm_Standard
+ fm_Terrace
+ Q("fm_Type S1")
+ Q("fm_Type S2")

+ ft_2
+ ft_3
+ ft_4
+ ft_5
+ ft_6
+ ft_7

+ C(storey_relative_category)
+ C(town)
+ C(year_quarter)
'''

formula_m3 = f'''
log_real_price_psf ~ {main_school_var} + {secondary_school_var}
+ floor_area_sqm
+ remaining_lease_years
+ mature_estate
+ dist_to_cbd_km
+ dist_to_nearest_mrt_km
+ dist_to_nearest_mall_km
+ dist_to_nearest_hawker_km
+ dist_to_nearest_busstop_km
+ countall_0_1km
+ countall_1_2km

+ fm_3Gen
+ Q("fm_Adjoined flat")
+ fm_Apartment
+ fm_DBSS
+ fm_Improved
+ Q("fm_Improved-Maisonette")
+ fm_Maisonette
+ Q("fm_Model A")
+ Q("fm_Model A-Maisonette")
+ Q("fm_Model A2")
+ Q("fm_New Generation")
+ Q("fm_Premium Apartment")
+ Q("fm_Premium Apartment Loft")
+ Q("fm_Premium Maisonette")
+ fm_Simplified
+ fm_Standard
+ fm_Terrace
+ Q("fm_Type S1")
+ Q("fm_Type S2")

+ ft_2
+ ft_3
+ ft_4
+ ft_5
+ ft_6
+ ft_7

+ C(storey_relative_category)
+ C(town)
+ C(year_quarter)
'''

model1 = smf.ols(formula=formula_m1, data=reg_df).fit(cov_type='HC3')
model2 = smf.ols(formula=formula_m2, data=reg_df).fit(cov_type='HC3')
model3 = smf.ols(formula=formula_m3, data=reg_df).fit(cov_type='HC3')

In [78]:
def pct_effect(beta):
    return (np.exp(beta) - 1) * 100


def summarize_key_terms(model, model_name):
    rows = []
    for var in [main_school_var, secondary_school_var]:
        beta = model.params[var]
        rows.append({
            'Model': model_name,
            'Variable': var,
            'Coef': beta,
            'SE': model.bse[var],
            'p_value': model.pvalues[var],
            'Pct_effect': pct_effect(beta),
            'Adj_R2': model.rsquared_adj,
            'N': int(model.nobs),
        })
    return pd.DataFrame(rows)

results_main = pd.concat([
    summarize_key_terms(model1, 'Model 1: FE only'),
    summarize_key_terms(model2, 'Model 2: + accessibility'),
    summarize_key_terms(model3, 'Model 3: + school density'),
], ignore_index=True)

results_main

,Model,Variable,Coef,SE,p_value,Pct_effect,Adj_R2,N
0,Model 1: FE only,d_0_1km_good80,0.0072,0.0004,0.0000,0.7268,0.7779,291419
1,Model 1: FE only,d_1_2km_good80,0.0164,0.0005,0.0000,1.6554,0.7779,291419
2,Model 2: + accessibility,d_0_1km_good80,0.0087,0.0004,0.0000,0.8726,0.8220,291419
3,Model 2: + accessibility,d_1_2km_good80,0.0075,0.0004,0.0000,0.7545,0.8220,291419
4,Model 3: + school density,d_0_1km_good80,0.0115,0.0004,0.0000,1.1599,0.8229,291419
5,Model 3: + school density,d_1_2km_good80,0.0041,0.0004,0.0000,0.4148,0.8229,291419


In [79]:
model3.summary().tables[1]

,coef,std err,z,P>|z|,[0.025,0.975]
Intercept,5.4148,0.010,527.670,0.000,5.395,5.435
fm_3Gen[T.True],-0.1223,0.012,-10.153,0.000,-0.146,-0.099
"Q(""fm_Adjoined flat"")[T.True]",0.1056,0.008,13.229,0.000,0.090,0.121
fm_Apartment[T.True],0.0493,0.006,7.900,0.000,0.037,0.061
fm_DBSS[T.True],0.1531,0.006,25.164,0.000,0.141,0.165
fm_Improved[T.True],-0.0318,0.006,-5.359,0.000,-0.043,-0.020
"Q(""fm_Improved-Maisonette"")[T.True]",0.2769,0.015,18.618,0.000,0.248,0.306
fm_Maisonette[T.True],0.0983,0.006,15.710,0.000,0.086,0.111
"Q(""fm_Model A"")[T.True]",-0.0331,0.006,-5.617,0.000,-0.045,-0.022
"Q(""fm_Model A-Maisonette"")[T.True]",0.1498,0.007,20.623,0.000,0.136,0.164


## Preferred interpretation

Read Model 3 as the main result. The coefficient on `d_0_1km_good80` is the conditional association between being within 1 km of a top-20% primary school and log real resale price per square foot, after controlling for observed flat characteristics, broad location, market timing, accessibility, and total nearby school density.

The coefficient on `d_1_2km_good80` captures the weaker association for homes in the 1 to 2 km ring.

In [80]:
preferred = pd.DataFrame({
    'term': [main_school_var, secondary_school_var],
    'coef': [model3.params[main_school_var], model3.params[secondary_school_var]],
    'std_err': [model3.bse[main_school_var], model3.bse[secondary_school_var]],
    'p_value': [model3.pvalues[main_school_var], model3.pvalues[secondary_school_var]],
})
preferred['pct_effect'] = preferred['coef'].apply(pct_effect)
preferred

,term,coef,std_err,p_value,pct_effect
0,d_0_1km_good80,0.0115,0.0004,0.0000,1.1599
1,d_1_2km_good80,0.0041,0.0004,0.0000,0.4148


## Clustered standard errors by town

This is a sensitivity check only. There are only 26 towns, so clustered inference at the town level is conservative and potentially unstable. Report it as a robustness check, not the main standard-error choice.

In [81]:
model3_cluster = smf.ols(formula=formula_m3, data=reg_df).fit(
    cov_type='cluster',
    cov_kwds={'groups': reg_df['town']}
)

cluster_compare = pd.DataFrame({
    'term': [main_school_var, secondary_school_var],
    'coef_hc3': [model3.params[main_school_var], model3.params[secondary_school_var]],
    'se_hc3': [model3.bse[main_school_var], model3.bse[secondary_school_var]],
    'p_hc3': [model3.pvalues[main_school_var], model3.pvalues[secondary_school_var]],
    'coef_cluster_town': [model3_cluster.params[main_school_var], model3_cluster.params[secondary_school_var]],
    'se_cluster_town': [model3_cluster.bse[main_school_var], model3_cluster.bse[secondary_school_var]],
    'p_cluster_town': [model3_cluster.pvalues[main_school_var], model3_cluster.pvalues[secondary_school_var]],
})

cluster_compare

,term,coef_hc3,se_hc3,p_hc3,coef_cluster_town,se_cluster_town,p_cluster_town
0,d_0_1km_good80,0.0115,0.0004,0.0000,0.0115,0.0077,0.1361
1,d_1_2km_good80,0.0041,0.0004,0.0000,0.0041,0.0076,0.5845


## Robustness 1: alternative school-quality thresholds

In [82]:
def run_threshold_model(threshold):
    main_var = f'd_0_1km_good{threshold}'
    sec_var = f'd_1_2km_good{threshold}'

    vars_needed = [
        'log_real_price_psf',
        main_var,
        sec_var,
        'countall_0_1km',
        'countall_1_2km',
        'floor_area_sqm',
        'remaining_lease_years',
        'mature_estate',
        'dist_to_cbd_km',
        'dist_to_nearest_mrt_km',
        'dist_to_nearest_mall_km',
        'dist_to_nearest_hawker_km',
        'dist_to_nearest_busstop_km',
        'town',
        'storey_relative_category',
        'year_quarter',
        'fm_3Gen',
        'fm_Adjoined flat',
        'fm_Apartment',
        'fm_DBSS',
        'fm_Improved',
        'fm_Improved-Maisonette',
        'fm_Maisonette',
        'fm_Model A',
        'fm_Model A-Maisonette',
        'fm_Model A2',
        'fm_New Generation',
        'fm_Premium Apartment',
        'fm_Premium Apartment Loft',
        'fm_Premium Maisonette',
        'fm_Simplified',
        'fm_Standard',
        'fm_Terrace',
        'fm_Type S1',
        'fm_Type S2',
        'ft_2', 'ft_3', 'ft_4', 'ft_5', 'ft_6', 'ft_7'
    ]

    temp_df = df_clean[vars_needed].dropna().copy()

    formula = f'''
    log_real_price_psf ~ {main_var} + {sec_var}
    + floor_area_sqm
    + remaining_lease_years
    + mature_estate
    + dist_to_cbd_km
    + dist_to_nearest_mrt_km
    + dist_to_nearest_mall_km
    + dist_to_nearest_hawker_km
    + dist_to_nearest_busstop_km
    + countall_0_1km
    + countall_1_2km

    + fm_3Gen
    + Q("fm_Adjoined flat")
    + fm_Apartment
    + fm_DBSS
    + fm_Improved
    + Q("fm_Improved-Maisonette")
    + fm_Maisonette
    + Q("fm_Model A")
    + Q("fm_Model A-Maisonette")
    + Q("fm_Model A2")
    + Q("fm_New Generation")
    + Q("fm_Premium Apartment")
    + Q("fm_Premium Apartment Loft")
    + Q("fm_Premium Maisonette")
    + fm_Simplified
    + fm_Standard
    + fm_Terrace
    + Q("fm_Type S1")
    + Q("fm_Type S2")

    + ft_2
    + ft_3
    + ft_4
    + ft_5
    + ft_6
    + ft_7

    + C(storey_relative_category)
    + C(town)
    + C(year_quarter)
    '''

    model = smf.ols(formula=formula, data=temp_df).fit(cov_type='HC3')

    beta_01 = model.params[main_var]
    beta_12 = model.params[sec_var]
    return {
        'threshold': threshold,
        'share_0_1km': temp_df[main_var].mean(),
        'share_1_2km': temp_df[sec_var].mean(),
        'coef_0_1km': beta_01,
        'pct_0_1km': pct_effect(beta_01),
        'se_0_1km': model.bse[main_var],
        'p_0_1km': model.pvalues[main_var],
        'coef_1_2km': beta_12,
        'pct_1_2km': pct_effect(beta_12),
        'se_1_2km': model.bse[sec_var],
        'p_1_2km': model.pvalues[sec_var],
        'Adj_R2': model.rsquared_adj,
        'N': int(model.nobs),
    }

threshold_results = pd.DataFrame([run_threshold_model(t) for t in [75, 80, 85, 90]])
threshold_results

,threshold,share_0_1km,share_1_2km,coef_0_1km,pct_0_1km,se_0_1km,p_0_1km,coef_1_2km,pct_1_2km,se_1_2km,p_1_2km,Adj_R2,N
0,75,0.5468,0.6986,0.0046,0.4597,0.0004,0.0000,0.0080,0.8006,0.0005,0.0000,0.8227,291419
1,80,0.4226,0.5794,0.0115,1.1599,0.0004,0.0000,0.0041,0.4148,0.0004,0.0000,0.8229,291419
2,85,0.3242,0.4752,0.0126,1.2686,0.0004,0.0000,0.0034,0.3356,0.0004,0.0000,0.8230,291419
3,90,0.2102,0.3465,0.0067,0.6684,0.0005,0.0000,0.0012,0.1244,0.0005,0.0062,0.8226,291419


## Robustness 2: count of good schools instead of a dummy

In [83]:
count_main_var = 'count_0_1km_good80'
count_sec_var = 'count_1_2km_good80'

count_vars = [
    'log_real_price_psf',
    count_main_var,
    count_sec_var,
    'countall_0_1km',
    'countall_1_2km',
    'floor_area_sqm',
    'remaining_lease_years',
    'mature_estate',
    'dist_to_cbd_km',
    'dist_to_nearest_mrt_km',
    'dist_to_nearest_mall_km',
    'dist_to_nearest_hawker_km',
    'dist_to_nearest_busstop_km',
    'town',
    'storey_relative_category',
    'year_quarter',

    'fm_3Gen',
    'fm_Adjoined flat',
    'fm_Apartment',
    'fm_DBSS',
    'fm_Improved',
    'fm_Improved-Maisonette',
    'fm_Maisonette',
    'fm_Model A',
    'fm_Model A-Maisonette',
    'fm_Model A2',
    'fm_New Generation',
    'fm_Premium Apartment',
    'fm_Premium Apartment Loft',
    'fm_Premium Maisonette',
    'fm_Simplified',
    'fm_Standard',
    'fm_Terrace',
    'fm_Type S1',
    'fm_Type S2',

    'ft_2', 'ft_3', 'ft_4', 'ft_5', 'ft_6', 'ft_7'
]

reg_count = df_clean[count_vars].dropna().copy()

formula_count = f'''
log_real_price_psf ~ {count_main_var} + {count_sec_var}
+ floor_area_sqm
+ remaining_lease_years
+ mature_estate
+ dist_to_cbd_km
+ dist_to_nearest_mrt_km
+ dist_to_nearest_mall_km
+ dist_to_nearest_hawker_km
+ dist_to_nearest_busstop_km
+ countall_0_1km
+ countall_1_2km

+ fm_3Gen
+ Q("fm_Adjoined flat")
+ fm_Apartment
+ fm_DBSS
+ fm_Improved
+ Q("fm_Improved-Maisonette")
+ fm_Maisonette
+ Q("fm_Model A")
+ Q("fm_Model A-Maisonette")
+ Q("fm_Model A2")
+ Q("fm_New Generation")
+ Q("fm_Premium Apartment")
+ Q("fm_Premium Apartment Loft")
+ Q("fm_Premium Maisonette")
+ fm_Simplified
+ fm_Standard
+ fm_Terrace
+ Q("fm_Type S1")
+ Q("fm_Type S2")

+ ft_2
+ ft_3
+ ft_4
+ ft_5
+ ft_6
+ ft_7

+ C(storey_relative_category)
+ C(town)
+ C(year_quarter)
'''

model_count = smf.ols(formula=formula_count, data=reg_count).fit(cov_type='HC3')

count_results = pd.DataFrame({
    'term': [count_main_var, count_sec_var],
    'coef': [model_count.params[count_main_var], model_count.params[count_sec_var]],
    'std_err': [model_count.bse[count_main_var], model_count.bse[count_sec_var]],
    'p_value': [model_count.pvalues[count_main_var], model_count.pvalues[count_sec_var]],
    'approx_pct_per_extra_school': [
        pct_effect(model_count.params[count_main_var]),
        pct_effect(model_count.params[count_sec_var]),
    ],
    'Adj_R2': [model_count.rsquared_adj, model_count.rsquared_adj],
    'AIC': [model_count.aic, model_count.aic],
    'BIC': [model_count.bic, model_count.bic],
    'N': [int(model_count.nobs), int(model_count.nobs)],
})

count_results

,term,coef,std_err,p_value,approx_pct_per_extra_school,Adj_R2,AIC,BIC,N
0,count_0_1km_good80,0.0078,0.0004,0.0000,0.7834,0.8228,"-563,803.5865","-562,586.5970",291419
1,count_1_2km_good80,0.0007,0.0003,0.0068,0.0729,0.8228,"-563,803.5865","-562,586.5970",291419


## Robustness 3: quadratic distance controls

This robustness check allows the accessibility-price relationship to be nonlinear by adding squared distance terms for CBD, MRT, malls, hawker centres, and bus stops. The preferred baseline remains the linear-distance Model 3.


In [84]:
reg_df_quad = reg_df.copy()

distance_vars = [
    'dist_to_cbd_km',
    'dist_to_nearest_mrt_km',
    'dist_to_nearest_mall_km',
    'dist_to_nearest_hawker_km',
    'dist_to_nearest_busstop_km',
]

for var in distance_vars:
    reg_df_quad[var.replace('_km', '_sq')] = reg_df_quad[var] ** 2

formula_m4 = f'''
log_real_price_psf ~ {main_school_var} + {secondary_school_var}
+ floor_area_sqm
+ remaining_lease_years
+ mature_estate
+ dist_to_cbd_km
+ dist_to_cbd_sq
+ dist_to_nearest_mrt_km
+ dist_to_nearest_mrt_sq
+ dist_to_nearest_mall_km
+ dist_to_nearest_mall_sq
+ dist_to_nearest_hawker_km
+ dist_to_nearest_hawker_sq
+ dist_to_nearest_busstop_km
+ dist_to_nearest_busstop_sq
+ countall_0_1km
+ countall_1_2km

+ fm_3Gen
+ Q("fm_Adjoined flat")
+ fm_Apartment
+ fm_DBSS
+ fm_Improved
+ Q("fm_Improved-Maisonette")
+ fm_Maisonette
+ Q("fm_Model A")
+ Q("fm_Model A-Maisonette")
+ Q("fm_Model A2")
+ Q("fm_New Generation")
+ Q("fm_Premium Apartment")
+ Q("fm_Premium Apartment Loft")
+ Q("fm_Premium Maisonette")
+ fm_Simplified
+ fm_Standard
+ fm_Terrace
+ Q("fm_Type S1")
+ Q("fm_Type S2")

+ ft_2
+ ft_3
+ ft_4
+ ft_5
+ ft_6
+ ft_7

+ C(storey_relative_category)
+ C(town)
+ C(year_quarter)
'''

model4 = smf.ols(formula=formula_m4, data=reg_df_quad).fit(cov_type='HC3')

quadratic_distance_compare = pd.DataFrame({
    'model': ['Model 3 linear distances', 'Model 4 quadratic distances'],
    'coef_0_1km': [model3.params[main_school_var], model4.params[main_school_var]],
    'pct_0_1km': [pct_effect(model3.params[main_school_var]), pct_effect(model4.params[main_school_var])],
    'se_0_1km': [model3.bse[main_school_var], model4.bse[main_school_var]],
    'p_0_1km': [model3.pvalues[main_school_var], model4.pvalues[main_school_var]],
    'coef_1_2km': [model3.params[secondary_school_var], model4.params[secondary_school_var]],
    'pct_1_2km': [pct_effect(model3.params[secondary_school_var]), pct_effect(model4.params[secondary_school_var])],
    'se_1_2km': [model3.bse[secondary_school_var], model4.bse[secondary_school_var]],
    'p_1_2km': [model3.pvalues[secondary_school_var], model4.pvalues[secondary_school_var]],
    'Adj_R2': [model3.rsquared_adj, model4.rsquared_adj],
    'AIC': [model3.aic, model4.aic],
    'BIC': [model3.bic, model4.bic],
    'N': [int(model3.nobs), int(model4.nobs)],
})

quadratic_distance_compare

,model,coef_0_1km,pct_0_1km,se_0_1km,p_0_1km,coef_1_2km,pct_1_2km,se_1_2km,p_1_2km,Adj_R2,AIC,BIC,N
0,Model 3 linear distances,0.0115,1.1599,0.0004,0.0000,0.0041,0.4148,0.0004,0.0000,0.8229,"-564,078.9423","-562,861.9528",291419
1,Model 4 quadratic distances,0.0148,1.4907,0.0004,0.0000,0.0061,0.6161,0.0004,0.0000,0.8245,"-566,716.4242","-565,446.5222",291419


## Robustness 4: restricted sample (4-room and 5-room flats)

The original notebook filtered on string labels (`"4 ROOM"`, `"5 ROOM"`), but `flat_type` is numerically encoded in this dataset. This version uses codes `4` and `5`.

In [85]:
restricted_df = df_clean.loc[df_clean['flat_type'].isin([4, 5])].copy()

restricted_vars = [
    'log_real_price_psf',
    main_school_var,
    secondary_school_var,
    'countall_0_1km',
    'countall_1_2km',
    'floor_area_sqm',
    'remaining_lease_years',
    'mature_estate',
    'dist_to_cbd_km',
    'dist_to_nearest_mrt_km',
    'dist_to_nearest_mall_km',
    'dist_to_nearest_hawker_km',
    'dist_to_nearest_busstop_km',
    'town',
    'storey_relative_category',
    'year_quarter',

    'fm_3Gen',
    'fm_Adjoined flat',
    'fm_Apartment',
    'fm_DBSS',
    'fm_Improved',
    'fm_Improved-Maisonette',
    'fm_Maisonette',
    'fm_Model A',
    'fm_Model A-Maisonette',
    'fm_Model A2',
    'fm_New Generation',
    'fm_Premium Apartment',
    'fm_Premium Apartment Loft',
    'fm_Premium Maisonette',
    'fm_Simplified',
    'fm_Standard',
    'fm_Terrace',
    'fm_Type S1',
    'fm_Type S2',

    'ft_5'
]

reg_restricted = restricted_df[restricted_vars].dropna().copy()
print('Restricted-sample observations:', reg_restricted.shape[0])

formula_restricted = f'''
log_real_price_psf ~ {main_school_var} + {secondary_school_var}
+ floor_area_sqm
+ remaining_lease_years
+ mature_estate
+ dist_to_cbd_km
+ dist_to_nearest_mrt_km
+ dist_to_nearest_mall_km
+ dist_to_nearest_hawker_km
+ dist_to_nearest_busstop_km
+ countall_0_1km
+ countall_1_2km

+ fm_3Gen
+ Q("fm_Adjoined flat")
+ fm_Apartment
+ fm_DBSS
+ fm_Improved
+ Q("fm_Improved-Maisonette")
+ fm_Maisonette
+ Q("fm_Model A")
+ Q("fm_Model A-Maisonette")
+ Q("fm_Model A2")
+ Q("fm_New Generation")
+ Q("fm_Premium Apartment")
+ Q("fm_Premium Apartment Loft")
+ Q("fm_Premium Maisonette")
+ fm_Simplified
+ fm_Standard
+ fm_Terrace
+ Q("fm_Type S1")
+ Q("fm_Type S2")

+ ft_5

+ C(storey_relative_category)
+ C(town)
+ C(year_quarter)
'''

model_restricted = smf.ols(formula=formula_restricted, data=reg_restricted).fit(cov_type='HC3')

restricted_results = pd.DataFrame({
    'term': [main_school_var, secondary_school_var],
    'coef': [model_restricted.params[main_school_var], model_restricted.params[secondary_school_var]],
    'std_err': [model_restricted.bse[main_school_var], model_restricted.bse[secondary_school_var]],
    'p_value': [model_restricted.pvalues[main_school_var], model_restricted.pvalues[secondary_school_var]],
    'pct_effect': [
        pct_effect(model_restricted.params[main_school_var]),
        pct_effect(model_restricted.params[secondary_school_var]),
    ],
    'Adj_R2': [model_restricted.rsquared_adj, model_restricted.rsquared_adj],
    'AIC': [model_restricted.aic, model_restricted.aic],
    'BIC': [model_restricted.bic, model_restricted.bic],
    'N': [int(model_restricted.nobs), int(model_restricted.nobs)],
})

restricted_results

Restricted-sample observations: 192067


,term,coef,std_err,p_value,pct_effect,Adj_R2,AIC,BIC,N
0,d_0_1km_good80,0.0094,0.0000,0.0000,0.9447,0.8711,"-404,481.3061","-403,403.7525",192067
1,d_1_2km_good80,0.0048,0.0000,0.0000,0.4768,0.8711,"-404,481.3061","-403,403.7525",192067


## Robustness 5: Weights in GSI

In [97]:
robust_weights_1 = pd.read_csv("../data/final_df_robust_1.csv")
robust_weights_2 = pd.read_csv("../data/final_df_robust_2.csv")


In [98]:
processed_dfs = []

for df in [robust_weights_1, robust_weights_2]:
    flat_model_dummies = pd.get_dummies(df["flat_model"], prefix="fm", drop_first=True)

    # Remove problematic column (only if exists)
    if "fm_Multi Generation" in flat_model_dummies.columns:
        flat_model_dummies = flat_model_dummies.drop(columns=["fm_Multi Generation"])

    flat_type_dummies = pd.get_dummies(df["flat_type"], prefix="ft", drop_first=True)

    df_clean = pd.concat([
        df,
        flat_type_dummies,
        flat_model_dummies
    ], axis=1)

    df_clean['year_quarter'] = df_clean['year'].astype(int).astype(str) + 'Q' + df_clean['quarter'].astype(int).astype(str)

    processed_dfs.append(df_clean)

processed_dfs.append(reg_df)

In [99]:
# =========================
# 1. Helper functions
# =========================

def significance_stars(p):
    if p < 0.01:
        return '***'
    elif p < 0.05:
        return '**'
    elif p < 0.10:
        return '*'
    return ''

def extract_key_results(model, main_var, secondary_var, weight_label):
    rows = []
    for var in [main_var, secondary_var]:
        coef = model.params[var]
        se = model.bse[var]
        pval = model.pvalues[var]

        rows.append({
            'weights': weight_label,
            'term': var,
            'coef': coef,
            'std_err': se,
            'p_value': pval,
            'pct_effect': pct_effect(coef),
            'Adj_R2': model.rsquared_adj,
            'AIC': model.aic,
            'BIC': model.bic,
            'N': int(model.nobs),
        })
    return pd.DataFrame(rows)

In [100]:
for df in processed_dfs:
    main_school_var = 'd_0_1km_good80'
    secondary_school_var = 'd_1_2km_good80'
    formula_m3 = f'''
log_real_price_psf ~ {main_school_var} + {secondary_school_var}
+ floor_area_sqm
+ remaining_lease_years
+ mature_estate
+ dist_to_cbd_km
+ dist_to_nearest_mrt_km
+ dist_to_nearest_mall_km
+ dist_to_nearest_hawker_km
+ dist_to_nearest_busstop_km
+ countall_0_1km
+ countall_1_2km

+ fm_3Gen
+ Q("fm_Adjoined flat")
+ fm_Apartment
+ fm_DBSS
+ fm_Improved
+ Q("fm_Improved-Maisonette")
+ fm_Maisonette
+ Q("fm_Model A")
+ Q("fm_Model A-Maisonette")
+ Q("fm_Model A2")
+ Q("fm_New Generation")
+ Q("fm_Premium Apartment")
+ Q("fm_Premium Apartment Loft")
+ Q("fm_Premium Maisonette")
+ fm_Simplified
+ fm_Standard
+ fm_Terrace
+ Q("fm_Type S1")
+ Q("fm_Type S2")

+ ft_2
+ ft_3
+ ft_4
+ ft_5
+ ft_6
+ ft_7

+ C(storey_relative_category)
+ C(town)
+ C(year_quarter)
'''
    model3 = smf.ols(formula=formula_m3, data=reg_df).fit(cov_type='HC3')

weight_labels = [
    '0.7SDI_0.15GEP_0.15SAP',
    '0.8SDI_0.1GEP_0.1SAP',
    '0.6SDI_0.2GEP_0.2SAP',
]

all_results = []
all_models = {}

for df, w in zip(processed_dfs, weight_labels):
    model = smf.ols(formula=formula_m3, data=df).fit(cov_type='HC3')
    all_models[w] = model

    result_df = extract_key_results(
        model=model,
        main_var=main_school_var,
        secondary_var=secondary_school_var,
        weight_label=w
    )
    all_results.append(result_df)

final_results = pd.concat(all_results, ignore_index=True)


print("\n=== Raw key results ===")
print(final_results)


=== Raw key results ===
                  weights            term   coef  std_err  p_value stars  \
0  0.7SDI_0.15GEP_0.15SAP  d_0_1km_good80 0.0116   0.0004   0.0000   ***   
1  0.7SDI_0.15GEP_0.15SAP  d_1_2km_good80 0.0050   0.0004   0.0000   ***   
2    0.8SDI_0.1GEP_0.1SAP  d_0_1km_good80 0.0108   0.0004   0.0000   ***   
3    0.8SDI_0.1GEP_0.1SAP  d_1_2km_good80 0.0049   0.0004   0.0000   ***   
4    0.6SDI_0.2GEP_0.2SAP  d_0_1km_good80 0.0115   0.0004   0.0000   ***   
5    0.6SDI_0.2GEP_0.2SAP  d_1_2km_good80 0.0041   0.0004   0.0000   ***   

  coef_with_stars  pct_effect  Adj_R2           AIC           BIC       N  
0       0.0116***      1.1688  0.8230 -564,119.6970 -562,902.7075  291419  
1       0.0050***      0.4964  0.8230 -564,119.6970 -562,902.7075  291419  
2       0.0108***      1.0894  0.8229 -564,044.9946 -562,828.0051  291419  
3       0.0049***      0.4935  0.8229 -564,044.9946 -562,828.0051  291419  
4       0.0115***      1.1599  0.8229 -564,078.9423 -562,861.9

In [101]:
final_results

,weights,term,coef,std_err,p_value,stars,coef_with_stars,pct_effect,Adj_R2,AIC,BIC,N
0,0.7SDI_0.15GEP_0.15SAP,d_0_1km_good80,0.0116,0.0004,0.0000,***,0.0116***,1.1688,0.8230,"-564,119.6970","-562,902.7075",291419
1,0.7SDI_0.15GEP_0.15SAP,d_1_2km_good80,0.0050,0.0004,0.0000,***,0.0050***,0.4964,0.8230,"-564,119.6970","-562,902.7075",291419
2,0.8SDI_0.1GEP_0.1SAP,d_0_1km_good80,0.0108,0.0004,0.0000,***,0.0108***,1.0894,0.8229,"-564,044.9946","-562,828.0051",291419
3,0.8SDI_0.1GEP_0.1SAP,d_1_2km_good80,0.0049,0.0004,0.0000,***,0.0049***,0.4935,0.8229,"-564,044.9946","-562,828.0051",291419
4,0.6SDI_0.2GEP_0.2SAP,d_0_1km_good80,0.0115,0.0004,0.0000,***,0.0115***,1.1599,0.8229,"-564,078.9423","-562,861.9528",291419
5,0.6SDI_0.2GEP_0.2SAP,d_1_2km_good80,0.0041,0.0004,0.0000,***,0.0041***,0.4148,0.8229,"-564,078.9423","-562,861.9528",291419


## Takeaways

- The preferred specification is Model 3.
- A `good80` school within `0-1km` is typically associated with about a 1% premium in real resale price per square foot.
- The `1-2km` association is smaller and should be described as economically modest.
- The result is robust to a quadratic distance specification; allowing nonlinear accessibility effects increases the estimated `0-1km` premium rather than removing it.
- Clustered-by-town standard errors materially weaken statistical significance, so inference is sensitive to how spatial correlation is handled.
- This remains a hedonic association, not a causal estimate of school quality on housing prices.
